# 01 — Exploratory Data Analysis: USGS NWIS Groundwater Sites

**Purpose**: Profile and visualize all downloaded NWIS groundwater monitoring sites before any QC or modeling.

Sections:
1. Load & concatenate all state site files
2. Data quality flags (datum, depth, coordinates)
3. Spatial coverage map
4. Site density by state
5. Well depth distribution
6. Land surface elevation distribution
7. Aquifer type and datum breakdown
8. Temporal coverage (construction / inventory dates)
9. Summary statistics table

> **Note**: Ground-level measurements (`data/raw/nwis/*_gwlevels.parquet`) are not yet available — the `make data` run only collected site metadata for most states. Level data will be explored in `02_gwlevels_eda.ipynb` once available.

In [ ]:
from __future__ import annotations

import pathlib
import warnings

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ---- Paths ----
RAW_DIR = pathlib.Path('../data/raw/nwis')

# ---- Style ----
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
CMAP_DENSITY = 'YlOrRd'
CMAP_DEPTH   = 'viridis_r'
CMAP_ELEV    = 'terrain'

## 1. Load all site files

In [ ]:
site_files = sorted(RAW_DIR.glob('*_sites.parquet'))
print(f'Found {len(site_files)} state site files')

dfs = []
for f in site_files:
    df = pd.read_parquet(f)
    df['state'] = f.stem.replace('_sites', '')
    dfs.append(df)

sites = pd.concat(dfs, ignore_index=True)
print(f'Total sites: {len(sites):,}')
print(f'States: {sites["state"].nunique()}')

In [ ]:
# ---- Unit conversions (feet → meters) ----
FT2M = 0.3048
sites['well_depth_m']  = sites['well_depth_va']  * FT2M
sites['alt_va_m']      = sites['alt_va']          * FT2M

# ---- Drop obviously bogus alt_va values (NWIS sentinel = -9999 ft) ----
sites.loc[sites['alt_va'] < -999, 'alt_va_m'] = np.nan

# ---- QC flag columns ----
sites['flag_ngvd29']       = sites['alt_datum_cd'] == 'NGVD29'
sites['flag_deep_well']    = sites['well_depth_m'] > (150)       # >150 m likely confined
sites['flag_no_alt']       = sites['alt_va_m'].isna()
sites['flag_no_coord']     = sites['dec_lat_va'].isna() | sites['dec_long_va'].isna()
sites['flag_any']          = (sites['flag_ngvd29'] |
                               sites['flag_deep_well'] |
                               sites['flag_no_alt'] |
                               sites['flag_no_coord'])

# ---- Sites that would pass QC ----
sites_ok = sites[~sites['flag_any'] & (sites['alt_datum_cd'] == 'NAVD88')].copy()

flag_summary = pd.DataFrame({
    'Flag': ['NGVD29 datum', 'Deep well (>150 m)', 'No elevation (alt_va)', 'No coordinates', 'Any flag'],
    'Count': [
        sites['flag_ngvd29'].sum(),
        sites['flag_deep_well'].sum(),
        sites['flag_no_alt'].sum(),
        sites['flag_no_coord'].sum(),
        sites['flag_any'].sum(),
    ]
})
flag_summary['Pct'] = (flag_summary['Count'] / len(sites) * 100).round(1)
print(flag_summary.to_string(index=False))
print(f'\nSites passing all QC flags (NAVD88, shallow, has elev+coord): {len(sites_ok):,} '
      f'({len(sites_ok)/len(sites)*100:.1f}%)')

## 2. QC Flag Summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
labels = flag_summary['Flag'].tolist()
counts = flag_summary['Count'].tolist()
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#3498db', '#2c3e50']
bars = ax.barh(labels, counts, color=colors, edgecolor='white', linewidth=0.5)
for bar, pct in zip(bars, flag_summary['Pct']):
    ax.text(bar.get_width() + len(sites)*0.005, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=9, color='#444')
ax.set_xlabel('Number of sites')
ax.set_title('QC flag counts across all downloaded NWIS sites', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
ax.axvline(len(sites), color='gray', linestyle='--', lw=0.8, label=f'Total = {len(sites):,}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 3. Spatial Coverage Maps

In [ ]:
# Convert to GeoDataFrame for easy spatial plotting
sites_valid_coord = sites.dropna(subset=['dec_lat_va', 'dec_long_va'])
gdf = gpd.GeoDataFrame(
    sites_valid_coord,
    geometry=gpd.points_from_xy(sites_valid_coord['dec_long_va'], sites_valid_coord['dec_lat_va']),
    crs='EPSG:4326'
)
gdf_5070 = gdf.to_crs('EPSG:5070')   # NAD83 CONUS Albers for analysis

# QC-passing sites only
sites_ok_coord = sites_ok.dropna(subset=['dec_lat_va', 'dec_long_va'])
gdf_ok = gpd.GeoDataFrame(
    sites_ok_coord,
    geometry=gpd.points_from_xy(sites_ok_coord['dec_long_va'], sites_ok_coord['dec_lat_va']),
    crs='EPSG:4326'
).to_crs('EPSG:5070')

print(f'Sites with valid coordinates: {len(gdf):,}')
print(f'QC-passing sites with coordinates: {len(gdf_ok):,}')

In [ ]:
# ---- Load US state boundaries (built-in naturalearth via geopandas) ----
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
conus = world[(world['continent'] == 'North America') & (world['iso_a3'] == 'USA')].to_crs('EPSG:5070')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (gdf_plot, title, color) in zip(axes, [
    (gdf_5070, f'All downloaded sites\n(n={len(gdf_5070):,})', '#2b7bb9'),
    (gdf_ok,   f'QC-passing sites (NAVD88, shallow, georeferenced)\n(n={len(gdf_ok):,})', '#27ae60'),
]):
    conus.plot(ax=ax, color='#f5f5f0', edgecolor='#cccccc', linewidth=0.5)
    ax.scatter(
        gdf_plot.geometry.x, gdf_plot.geometry.y,
        s=0.3, alpha=0.3, color=color, linewidths=0, rasterized=True
    )
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_axis_off()

plt.suptitle('USGS NWIS Groundwater Monitoring Sites — CONUS', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Hexbin density map of QC-passing sites ----
fig, ax = plt.subplots(figsize=(14, 7))
conus.plot(ax=ax, color='#f0f0eb', edgecolor='#999', linewidth=0.5, zorder=1)

x = gdf_ok.geometry.x.values
y = gdf_ok.geometry.y.values
hb = ax.hexbin(x, y, gridsize=80, cmap=CMAP_DENSITY,
               norm=LogNorm(vmin=1, vmax=None),
               mincnt=1, linewidths=0, zorder=2)

cb = fig.colorbar(hb, ax=ax, shrink=0.6, pad=0.01)
cb.set_label('Sites per hex cell (log scale)', fontsize=10)
ax.set_title('Site density — QC-passing NWIS groundwater wells (NAVD88, ≤150 m)',
             fontsize=12, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 4. Site Count by State

In [ ]:
state_counts = (
    sites.groupby('state')
    .agg(
        total=('site_no', 'count'),
        navd88=('flag_ngvd29', lambda x: (~x).sum()),
        deep=('flag_deep_well', 'sum'),
    )
    .assign(pct_navd88=lambda d: d['navd88'] / d['total'] * 100)
    .sort_values('total', ascending=True)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 10))

# Left: total sites per state
ax = axes[0]
bars = ax.barh(state_counts.index, state_counts['total'],
               color='#2b7bb9', edgecolor='white', linewidth=0.3)
ax.set_xlabel('Number of sites')
ax.set_title('Total sites per state', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

# Right: % NAVD88 per state
ax = axes[1]
colors_pct = ['#27ae60' if p >= 50 else '#e74c3c' for p in state_counts['pct_navd88']]
ax.barh(state_counts.index, state_counts['pct_navd88'],
        color=colors_pct, edgecolor='white', linewidth=0.3)
ax.axvline(50, color='#555', linestyle='--', lw=1)
ax.set_xlabel('% sites with NAVD88 datum')
ax.set_title('Fraction of sites with NAVD88 elevation datum\n(green ≥ 50%, red < 50%)', fontweight='bold')

plt.suptitle('NWIS Groundwater Sites by State', fontsize=13, fontweight='bold', x=0.5, y=1.01)
plt.tight_layout()
plt.show()

## 5. Well Depth Distribution

In [ ]:
depth_valid = sites['well_depth_m'].dropna()
depth_valid = depth_valid[(depth_valid > 0) & (depth_valid < 5000)]  # drop clearly bogus values

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Linear scale
ax = axes[0]
ax.hist(depth_valid, bins=100, color='#2b7bb9', edgecolor='white', linewidth=0.2)
ax.axvline(150, color='#e74c3c', linestyle='--', lw=1.5, label='150 m (confined flag)')
ax.set_xlabel('Well depth (m)')
ax.set_ylabel('Number of sites')
ax.set_title('All depths (linear scale)')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

# Log scale
ax = axes[1]
ax.hist(depth_valid, bins=np.logspace(np.log10(0.1), np.log10(5000), 80),
        color='#8e44ad', edgecolor='white', linewidth=0.2)
ax.set_xscale('log')
ax.axvline(150, color='#e74c3c', linestyle='--', lw=1.5, label='150 m')
ax.set_xlabel('Well depth (m, log scale)')
ax.set_title('Log-scale depth distribution')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

# CDF
ax = axes[2]
sorted_d = np.sort(depth_valid)
cdf = np.arange(1, len(sorted_d)+1) / len(sorted_d)
ax.plot(sorted_d, cdf, color='#2c3e50', lw=1.5)
ax.axvline(150, color='#e74c3c', linestyle='--', lw=1.5, label='150 m')
p_shallow = (depth_valid <= 150).mean() * 100
ax.axhline(p_shallow/100, color='#e74c3c', linestyle=':', lw=1)
ax.text(155, p_shallow/100 + 0.02, f'{p_shallow:.1f}% ≤ 150 m', fontsize=9, color='#e74c3c')
ax.set_xlabel('Well depth (m)')
ax.set_ylabel('Cumulative fraction')
ax.set_title('CDF of well depth')
ax.legend(fontsize=9)
ax.set_xlim(0, 800)

plt.suptitle(f'Well Depth Distribution  (n={len(depth_valid):,} sites with depth record)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\nWell depth summary (meters):')
print(depth_valid.describe().rename(index=lambda x: x.capitalize()).to_frame('depth_m').T.to_string())
print(f'Sites with depth > 150 m (likely confined): {(depth_valid>150).sum():,} ({(depth_valid>150).mean()*100:.1f}%)')

## 6. Land Surface Elevation Distribution

In [ ]:
alt_valid = sites['alt_va_m'].dropna()
alt_valid = alt_valid[(alt_valid > -200) & (alt_valid < 4500)]  # plausible CONUS range

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.hist(alt_valid, bins=120, color='#27ae60', edgecolor='white', linewidth=0.2)
ax.set_xlabel('Land surface elevation (m, NAVD88)')
ax.set_ylabel('Number of sites')
ax.set_title('Land surface elevation of all sites')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

# Elevation vs latitude scatter (subsample for speed)
ax = axes[1]
sample = sites[['dec_lat_va', 'alt_va_m']].dropna()
sample = sample[(sample['alt_va_m'] > -200) & (sample['alt_va_m'] < 4500)]
idx = np.random.default_rng(42).choice(len(sample), min(80_000, len(sample)), replace=False)
ax.scatter(sample.iloc[idx]['dec_lat_va'], sample.iloc[idx]['alt_va_m'],
           s=0.5, alpha=0.15, color='#16a085', linewidths=0, rasterized=True)
ax.set_xlabel('Latitude (°N)')
ax.set_ylabel('Land surface elevation (m)')
ax.set_title('Elevation vs. latitude (random 80k subsample)')

plt.suptitle(f'Land Surface Elevation (n={len(alt_valid):,} sites with NAVD88/elevation)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Aquifer Type and Datum Breakdown

In [ ]:
AQFR_TYPE_LABELS = {
    'U': 'Unconfined',
    'C': 'Confined',
    'X': 'Mixed',
    'M': 'Multiple',
    'N': 'Not classified',
}
DATUM_COLORS = {'NAVD88': '#27ae60', 'NGVD29': '#e74c3c', 'LMSL': '#f39c12', 'NaN': '#95a5a6'}

fig = plt.figure(figsize=(15, 5))
gs = GridSpec(1, 3, figure=fig, wspace=0.4)

# -- Pie: datum distribution --
ax1 = fig.add_subplot(gs[0])
datum_counts = sites['alt_datum_cd'].fillna('NaN').value_counts()
colors_pie = [DATUM_COLORS.get(k, '#bdc3c7') for k in datum_counts.index]
wedges, texts, autotexts = ax1.pie(
    datum_counts.values, labels=datum_counts.index,
    colors=colors_pie, autopct='%1.1f%%',
    startangle=90, pctdistance=0.7,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for t in autotexts:
    t.set_fontsize(9)
ax1.set_title('Elevation datum\n(all sites)', fontweight='bold')

# -- Bar: aquifer type --
ax2 = fig.add_subplot(gs[1])
aqfr_counts = sites['aqfr_type_cd'].value_counts()
labels_aqfr = [AQFR_TYPE_LABELS.get(k, k) for k in aqfr_counts.index]
ax2.bar(labels_aqfr, aqfr_counts.values,
        color=['#3498db', '#e74c3c', '#f39c12', '#9b59b6', '#95a5a6'],
        edgecolor='white')
ax2.set_ylabel('Number of sites')
ax2.set_title('Aquifer type code\n(~16% of all sites have this field)', fontweight='bold')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
ax2.tick_params(axis='x', labelsize=9)

# -- Bar: top 10 national aquifer codes --
ax3 = fig.add_subplot(gs[2])
NAT_AQFR_LABELS = {
    'N100GLCIAL': 'Glacial',
    'N100HGHPLN': 'High Plains',
    'S100CSLLWD': 'Coastal Lowlands',
    'N100MSRVVL': 'Mississippi River Valley',
    'N9999OTHER': 'Other',
    'S100MSEMBM': 'Mississippi Embayment',
    'S100NATLCP': 'North Atlantic Coastal Plain',
    'N400PDMBRX': 'Piedmont & Blue Ridge',
    'N100BSNRGB': 'Basin and Range',
    'N100ALLUVL': 'Alluvial',
}
nat_counts = sites['nat_aqfr_cd'].value_counts().head(10)
nat_labels = [NAT_AQFR_LABELS.get(k, k) for k in nat_counts.index]
ax3.barh(nat_labels[::-1], nat_counts.values[::-1],
         color=plt.cm.tab10(np.linspace(0, 1, 10)), edgecolor='white')
ax3.set_xlabel('Number of sites')
ax3.set_title('Top 10 national aquifer codes\n(~50% of sites have this field)', fontweight='bold')
ax3.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
ax3.tick_params(axis='y', labelsize=8)

plt.suptitle('Aquifer Characterization and Datum Distribution', fontsize=13, fontweight='bold', y=1.02)
plt.show()

## 8. Spatial Map Colored by Well Depth and Elevation

In [ ]:
# Subsample QC-passing sites to a manageable scatter for coloring
gdf_plot = gdf_ok.dropna(subset=['well_depth_m', 'alt_va_m']).copy()
gdf_plot = gdf_plot[(gdf_plot['well_depth_m'] > 0) & (gdf_plot['well_depth_m'] < 500)]
rng = np.random.default_rng(0)
idx = rng.choice(len(gdf_plot), min(150_000, len(gdf_plot)), replace=False)
gdf_s = gdf_plot.iloc[idx]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, col, label, cmap, vmin, vmax in [
    (axes[0], 'well_depth_m', 'Well depth (m)', CMAP_DEPTH, 0, 300),
    (axes[1], 'alt_va_m', 'Land surface elevation (m NAVD88)', CMAP_ELEV, 0, 3000),
]:
    conus.plot(ax=ax, color='#e8e8e3', edgecolor='#aaa', linewidth=0.5, zorder=1)
    sc = ax.scatter(
        gdf_s.geometry.x, gdf_s.geometry.y,
        c=gdf_s[col], s=0.8, alpha=0.6,
        cmap=cmap, vmin=vmin, vmax=vmax,
        linewidths=0, rasterized=True, zorder=2
    )
    cb = fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.01)
    cb.set_label(label, fontsize=9)
    ax.set_title(label, fontweight='bold')
    ax.set_axis_off()

plt.suptitle(f'QC-passing NWIS sites (n={len(gdf_s):,} subsample) — Well Depth & Elevation',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 9. Well Construction Date (Temporal Coverage)

In [ ]:
# construction_dt is stored as YYYYMMDD float
constr = sites['construction_dt'].dropna()
constr = constr[(constr >= 19000101) & (constr <= 20260101)]
constr_year = (constr // 10000).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.hist(constr_year, bins=np.arange(1900, 2027, 1),
        color='#2980b9', edgecolor='white', linewidth=0.1)
ax.set_xlabel('Year')
ax.set_ylabel('Number of wells constructed')
ax.set_title('Well construction date (all years)', fontweight='bold')
ax.axvline(2000, color='#e74c3c', linestyle='--', lw=1.5, label='2000 (analysis start)')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

# Zoom to 1950–2026
ax = axes[1]
recent = constr_year[constr_year >= 1950]
ax.hist(recent, bins=np.arange(1950, 2027, 1),
        color='#8e44ad', edgecolor='white', linewidth=0.1)
ax.set_xlabel('Year')
ax.set_title('Well construction date (1950–present)', fontweight='bold')
ax.axvline(2000, color='#e74c3c', linestyle='--', lw=1.5, label='2000 (analysis start)')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

plt.suptitle(f'Well Construction Dates (n={len(constr_year):,} sites with construction date\n'
             f'= {len(constr_year)/len(sites)*100:.1f}% of all sites)',
             fontsize=12, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

## 10. Summary Statistics Table

In [ ]:
summary = pd.DataFrame([
    ('Total sites downloaded', f'{len(sites):,}', '49 states'),
    ('States / territories', f'{sites["state"].nunique()}', ''),
    ('Sites with coordinates', f'{gdf.shape[0]:,}', f'{gdf.shape[0]/len(sites)*100:.1f}%'),
    ('NAVD88 datum', f'{(sites["alt_datum_cd"]=="NAVD88").sum():,}',
      f'{(sites["alt_datum_cd"]=="NAVD88").mean()*100:.1f}%'),
    ('NGVD29 datum (excluded by default)', f'{(sites["alt_datum_cd"]=="NGVD29").sum():,}',
      f'{(sites["alt_datum_cd"]=="NGVD29").mean()*100:.1f}%'),
    ('Deep wells >150 m (likely confined)', f'{(sites["well_depth_m"]>150).sum():,}',
      f'{(sites["well_depth_m"]>150).mean()*100:.1f}%'),
    ('Sites passing all QC flags', f'{len(sites_ok):,}',
      f'{len(sites_ok)/len(sites)*100:.1f}%'),
    ('Median well depth (all)', f'{sites["well_depth_m"].median():.1f} m', ''),
    ('Median land surface elev (all)', f'{sites["alt_va_m"].median():.1f} m NAVD88', ''),
    ('Wells with aquifer type code', f'{sites["aqfr_type_cd"].notna().sum():,}',
      f'{sites["aqfr_type_cd"].notna().mean()*100:.1f}%'),
    ('Wells with national aquifer code', f'{sites["nat_aqfr_cd"].notna().sum():,}',
      f'{sites["nat_aqfr_cd"].notna().mean()*100:.1f}%'),
], columns=['Metric', 'Value', 'Note'])

print(summary.to_string(index=False))

## 11. Key Observations for Modeling

Based on this EDA:

| Finding | Impact | Action |
|---------|--------|--------|
| ~71% of sites have NGVD29 datum | **High** — cannot compute WTE without valid NAVD88 altitude | Run VERTCON correction or exclude; see `docs/assumptions.md` A4 |
| Only ~16% of sites have aquifer type code | **Medium** | Use `nat_aqfr_cd` (50% coverage) + well depth threshold as proxy |
| ~8% of sites have depth > 150 m | **Medium** — likely sampling confined aquifers | Flag as `confined_candidate`; use as covariate, not target |
| Site density is very uneven (MN: 92k, DC: few hundred) | **High** for spatial modeling | Well-density confidence mask is essential |
| `alt_va` has sentinel value −9999 (dropped above) | **Low** | Already handled in QC |
| Ground-level measurements not yet downloaded (0 `*_gwlevels.parquet` files) | **High** | `make data` run only fetched site metadata for most states — re-run with full level data flag |

> **Next step**: Run `make qc` once gwlevels parquets are available, then open `02_gwlevels_eda.ipynb` for temporal coverage and DTW/WTE distribution analysis.